<a href="https://colab.research.google.com/github/whiplaaash/TF-NLP/blob/main/colab_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP fundamentals in Tensor Flow

NLP has the goal of deriving information out of natural language (could be sequences, text or speach).

Another common term for NLP problems is sequence to seqence (seq2seq).

# Check for GPU (to change from CPU to GPU use "Srodowisko wykonawcze" and then "Zmien typ srodowiska wykonawczego").


In [ ]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-61bc4498-99b0-93ab-5ed7-ef9836a16db1)


# Get helper functions

In [ ]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py

--2026-11-09 21:30:43--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-11-09 21:30:44 (115 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Import a series of helper functions for the notebook.

In [ ]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

# Get a dataset

We are going to be using a dataset taken from Kaggle. It is an introduction to NLP dataset (text samples of Tweets labelled as a disaster or not a disaster).

Se the original source:
https://www.kaggle.com/competitions/nlp-getting-started

In [ ]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip


--2026-11-09 21:30:48--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.170.207, 74.125.200.207, 74.125.130.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.170.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K   660KB/s    in 0.9s    

2026-11-09 21:30:49 (660 KB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [ ]:
# Unzip data
unzip_data("nlp_getting_started.zip")

# Visualizing a text dataset

In [ ]:
# We use pandas package to read the data.

import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [ ]:
# Shuffle training dataframe

train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [ ]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [ ]:
# How many examples of class are there?
train_df_shuffled.target.value_counts()

# Data are quite well balanced (60:40)

,count
target,
0,4342
1,3271


In [ ]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

# Let's visualize some data

In [ ]:
# Let's generate 10 random samples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
# we generate random samples in the range [random_index:random_index+5]
# itertuples() - returns values in the row as a tuple
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target==1 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("-----\n")

Target: 0 (not real disaster)
Text:
Dr Jack Stern Interview Ending Back Pain for #Military #Injury. Listen now: http://t.co/YhH7X0MAio

-----

Target: 0 (not real disaster)
Text:
Owner of Chicago-Area Gay Bar Admits to Arson Scheme http://t.co/2Y9dnP5vtg via @theadvocatemag #LGBT | https://t.co/6XuL6DCOsh

-----

Target: 1 (real disaster)
Text:
Another fake hate crime Lesbians burn their own house down. What else Is new :http://t.co/66oBQmxImb

-----

Target: 1 (real disaster)
Text:
@BoyInAHorsemask its a panda trapped in a dogs body

-----

Target: 1 (real disaster)
Text:
@HighQualityBird a reverse situation (lol I don't know 9/11?) where US civilian deaths were specifically utilized to make a political

-----



In [ ]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1, # use 10% of data as validation set
                                                                            random_state=42 # for reproduction of the data
                                                                            )

In [ ]:
# Check lenghts
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

(6851, 762, 6851, 762)

In [ ]:
# check the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Converting text to numbers (tokenization and embbeding)

Before building a model based on words, the first thing to do is to convert words to numbers.

**Tokenization**: converting tokens (words, subwords, characters) into numbers, usually with one-hot encoding.

**Embedding**: takes into account the context of w word, probability of beng close to other words. We may use pre-learnt embedding. For each token you create a matrix of feature vectors.

# Tokenization (text vectorization)

In [ ]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Use the default parameters
text_vectorizer = TextVectorization(max_tokens=None, # how many tokens in the whole set of vocabulary
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None, # groups of n number of words
                                    output_mode="int", # how to represent words as numbers
                                    output_sequence_length=None, # how long do you want your sequences to be
                                    pad_to_max_tokens=False)

In [ ]:
# Find the avergae number of tokens in the training tweets
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [ ]:
# Setup text vectorization variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 15 # max length our sentences will be

# Use fixed, and no default, parameters
text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [ ]:
# Fit the text vectorizer to the training data
text_vectorizer.adapt(train_sentences)

In [ ]:
# Create a sample sentence and tokenize it
sample_sentence = "There is a flood on my street!"
text_vectorizer([sample_sentence])


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 74,   9,   3, 232,  11,  13, 698,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [ ]:
# choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
      \n\nVectorized version:")
text_vectorizer([random_sentence])

Original sentence:
Cool Tips from our friends at Counterstrike Security &amp; Sound.... http://t.co/z5Y4Xr14W6      

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 607, 1579,   20,  103,  819,   17,    1,  531,   35,  566,    1,
           0,    0,    0,    0]])>

In [ ]:
# Get the unique words in vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # get all the unique words in the vocabulary
top_5_words = words_in_vocab[:5] # get the most common words
bottom_5_words = words_in_vocab[-5:] # get the least common words in vocab
print(f"Number of words in vocabulary: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")


Number of words in vocabulary: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


# Embedding

To make an embedding we are going to use Tensor Flow embedding layer.

The parameters we care about most:
- `input_dim` - how many words are in vocabulary
- `output_dim` - the size of the output embedding vector, for example: a numbe 100 means that each token will be represented by a vector 100 long
- `input_length` - length of the sequences being passed to the embedding layer (this parameter no longer exist!)

In [ ]:
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length, #set the input shape
                             output_dim=128 # something divisible by 8, output shape
                             )
embedding

<Embedding name=embedding_1, built=False>

In [ ]:
# Get a random sentence from the trainig set
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
\n\nEmbedded version:")

# Embed a random sentence (turn it into dense vectors of fixed size)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original sentence:
December 2011 court dismissed the group charge of rape victims saying 'getting raped was an occupational hazard..!'

US military = ISIS!

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.03282105, -0.03200084,  0.00395579, ...,  0.01322222,
         -0.04318817, -0.0488433 ],
        [-0.00135058,  0.01394783,  0.02043606, ..., -0.02663674,
         -0.02959443,  0.00674538],
        [-0.04708202,  0.04844033,  0.00208819, ..., -0.03947805,
         -0.01745705,  0.03399631],
        ...,
        [-0.00994939,  0.04090405,  0.01083485, ..., -0.02618463,
          0.01018178,  0.03350442],
        [-0.00271614, -0.02156068, -0.04806351, ...,  0.03271437,
          0.01920014,  0.02806094],
        [-0.0265041 ,  0.02268546,  0.02013976, ...,  0.04007623,
          0.03295842, -0.04695028]]], shape=(1, 15, 128), dtype=float32)>

In [ ]:
# Check out a single token's embedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence[0]

# This array represents the first word in the random sentence
# The array shows random numbers which will be updated when the model learns
# The length of each suach array is 128 and is fixed

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([ 0.03282105, -0.03200084,  0.00395579, -0.03010012,  0.00526068,
         0.02882174, -0.0363246 , -0.00307661,  0.00212357,  0.04885379,
        -0.04652747,  0.04233812, -0.00972631,  0.04199222,  0.02858293,
        -0.02482584,  0.03454899,  0.01174313,  0.00499212, -0.00149137,
         0.02889123,  0.04710275, -0.00670927, -0.00966414, -0.02714266,
        -0.02517788, -0.01182334, -0.04743378, -0.02621528,  0.03002216,
        -0.04854058,  0.04177542, -0.03045871,  0.0105269 , -0.0456127 ,
        -0.02512833, -0.04069937,  0.0142994 ,  0.00370946, -0.01310455,
        -0.01516446,  0.01800776,  0.00489425,  0.01340941, -0.04584817,
        -0.01231688,  0.04138872,  0.01305002, -0.0033501 , -0.03455867,
         0.04816401, -0.02702077, -0.03842642, -0.041541  ,  0.0260679 ,
         0.00529281,  0.00106397,  0.03618205,  0.04153427, -0.01009289,
        -0.04415288,  0.00104795,  0.01960412, -0.00383887, -0.00688121,
  

# Modelling a text dataset (running a series of experiments)

After turning a text into numbers we will be running 8 experiments. We will start with a baseline and will be improving it.

* Model 0: Naive Bayes
* Model 1: Feed-forward neural network (dense model)
* Model 2: LSTM model (RNN)
* Model 3: GRU model (RNN)
* Model 4: Bidirectional LSTM model (RNN)
* Model 5: 1D Convolutional Neural Network model (CNN)
* Model 6: Tensor Flow Hub Pretrined Feature Extractor (using transfer learning for NLP)
* Model 7: Same as model 6 but with 10% of training data

To do all this we will go through the standard procedure of machine  learning:
- create a model
- build a model
- fit a model
- evaluate a model


# Model 0: Naive Bayes

This is a non-deep-learning model. It is part of Machine Learning and it comes from sklearn package.

⭕ It is important to start with a baseline model and then improve it. We will start with a Multinomial Naive Bayes with TF-IDF formula.

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [54]:
# Create tokanization and modelling pipeline
model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
    ("clf", MultinomialNB()) # model the text, "clf" means a classifier
])

# Fir the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [55]:
# Evaluate our baseline model
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 79.27%


This accuracy is more than a guess, which would be around 55%.

In [57]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [58]:
train_labels

array([0, 0, 1, ..., 1, 1, 0], shape=(6851,))

# Create a function which will be tracking the most important metrics across all models

Metrics:
- accuracy (not the best metric for imbalanced classes)
- precision (higher precison leads to less false positives)
- recall (higher recall leads to less false negatives)
- F1 score (combination of precision and recall)

In [59]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# similarly one could use tensorflow metrics instead of sklearn metrics

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall, and f1-score of a binary classification model.
  """
  # Calculate model accuracy:
  model_accuracy = accuracy_score(y_true, y_pred)*100
  # Calculate model precision, recall, f1-score using weighted average (weighted by support, that is, the number of cases in each class)
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  # the empty underscore "_" above stands for the support, but we are not interested in it
  model_results = {
      "accuracy": model_accuracy,
      "precision": model_precision,
      "recall": model_recall,
      "f1-score": model_f1}
  return model_results


In [60]:
# Get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}